### **Purpose:** Leakage-Safe Feature Engineering

This step creates rolling statistics, lag features, and short-term trend proxies while grouping by `season_year` to prevent information leakage across yearly July–October seasonal gaps.

In [400]:
# These columns were removed based on EDA findings
columns_to_drop = [
    "sst_cameroon_fraction_of_sea_ice_covered_ocean",   # constant value
    "sst_indian_ocean_fraction_of_sea_ice_covered_ocean",  # constant value
    "sst_cameroon_mean_temperature_kelvin",  # redundant with Celsius column
    "sst_indian_ocean_mean_temperature_kelvin",  # redundant with Celsius column
    "surface_pressure"  # almost identical to sea_level_pressure
]

In [401]:
train_clean_df = train_merged.drop(columns=columns_to_drop, errors="ignore")
test_clean_df = test_merged.drop(columns=columns_to_drop, errors="ignore")

In [402]:
print("Train shape", train_clean_df.shape)
print("Test shape", test_clean_df.shape)

print("\nRemaining columns:")
print(train_clean_df.columns.tolist())

Train shape (2214, 16)
Test shape (738, 15)

Remaining columns:
['date', '5cm_soil_moist', 'mean_dew_point_temp', 'max_temp', 'sea_level_pressure', 'sst_cameroon_mean_temperature_deg_c', 'sst_cameroon_mean_temperature_uncertainty', 'sst_indian_ocean_mean_temperature_deg_c', 'sst_indian_ocean_mean_temperature_uncertainty', 'potential_water_deficit', '2m_temp', 'u10', 'vapor_pressure_deficit', 'dryspell_warn_7d', 'year', 'month']


In [403]:
print("Train shape", train_merged.shape)
print("Test shape", test_merged.shape)

print("\nRemaining columns:")
print(train_merged.columns.tolist())

Train shape (2214, 21)
Test shape (738, 20)

Remaining columns:
['date', '5cm_soil_moist', 'mean_dew_point_temp', 'max_temp', 'sea_level_pressure', 'sst_cameroon_mean_temperature_kelvin', 'sst_cameroon_mean_temperature_deg_c', 'sst_cameroon_mean_temperature_uncertainty', 'sst_cameroon_fraction_of_sea_ice_covered_ocean', 'sst_indian_ocean_mean_temperature_kelvin', 'sst_indian_ocean_mean_temperature_deg_c', 'sst_indian_ocean_mean_temperature_uncertainty', 'sst_indian_ocean_fraction_of_sea_ice_covered_ocean', 'potential_water_deficit', 'surface_pressure', '2m_temp', 'u10', 'vapor_pressure_deficit', 'dryspell_warn_7d', 'year', 'month']


In [404]:
print("\nMissing values in train:")
print(train_clean_df.isna().sum().sort_values(ascending=False).head())

print("\nMissing values in test:")
print(test_clean_df.isna().sum().sort_values(ascending=False).head())




Missing values in train:
date                   0
5cm_soil_moist         0
mean_dew_point_temp    0
max_temp               0
sea_level_pressure     0
dtype: int64

Missing values in test:
sst_cameroon_mean_temperature_deg_c              123
sst_cameroon_mean_temperature_uncertainty        123
sst_indian_ocean_mean_temperature_deg_c          123
sst_indian_ocean_mean_temperature_uncertainty    123
mean_dew_point_temp                                1
dtype: int64


In [405]:
print("=== TEST DATA MISSINGNESS ===")
# % missing per column
missing_pct = test_merged.isna().mean() * 100
missing_cols = missing_pct[missing_pct > 0].sort_values(ascending=False)

# Rows with ANY missing data
rows_with_missing = test_merged.isna().any(axis=1).sum()
pct_rows_missing = (rows_with_missing / len(test_merged)) * 100

if len(missing_cols) == 0:
    print("Columns with missing data: None! 100% complete.")
else:
    print(f"Columns with missing data:\n{missing_cols}")
print(f"Rows with at least one missing value: {rows_with_missing} ({pct_rows_missing:.2f}%)\n")

=== TEST DATA MISSINGNESS ===
Columns with missing data:
sst_cameroon_mean_temperature_kelvin                  16.67
sst_cameroon_mean_temperature_deg_c                   16.67
sst_cameroon_mean_temperature_uncertainty             16.67
sst_cameroon_fraction_of_sea_ice_covered_ocean        16.67
sst_indian_ocean_mean_temperature_kelvin              16.67
sst_indian_ocean_mean_temperature_deg_c               16.67
sst_indian_ocean_mean_temperature_uncertainty         16.67
sst_indian_ocean_fraction_of_sea_ice_covered_ocean    16.67
mean_dew_point_temp                                    0.14
max_temp                                               0.14
sea_level_pressure                                     0.14
potential_water_deficit                                0.14
surface_pressure                                       0.14
2m_temp                                                0.14
dtype: float64
Rows with at least one missing value: 124 (16.80%)



In [406]:
print("=== TEST DATA MISSINGNESS ===")
# % missing per column
missing_pct = test_clean_df.isna().mean() * 100
missing_cols = missing_pct[missing_pct > 0].sort_values(ascending=False)

# Rows with ANY missing data
rows_with_missing = test_clean_df.isna().any(axis=1).sum()
pct_rows_missing = (rows_with_missing / len(test_clean_df)) * 100

if len(missing_cols) == 0:
    print("Columns with missing data: None! 100% complete.")
else:
    print(f"Columns with missing data:\n{missing_cols}")
print(f"Rows with at least one missing value: {rows_with_missing} ({pct_rows_missing:.2f}%)\n")

=== TEST DATA MISSINGNESS ===
Columns with missing data:
sst_cameroon_mean_temperature_deg_c              16.67
sst_cameroon_mean_temperature_uncertainty        16.67
sst_indian_ocean_mean_temperature_deg_c          16.67
sst_indian_ocean_mean_temperature_uncertainty    16.67
mean_dew_point_temp                               0.14
max_temp                                          0.14
sea_level_pressure                                0.14
potential_water_deficit                           0.14
2m_temp                                           0.14
dtype: float64
Rows with at least one missing value: 124 (16.80%)



In [407]:
train_clean_df.to_csv(processed_dir / "train_merged_clean.csv", index=False)
test_clean_df.to_csv(processed_dir / "test_merged_clean.csv", index=False)
print("Saved to:", processed_dir / "train_merged_clean.csv", "and", processed_dir / "test_merged_clean.csv")

Saved to: ../data/processed/train_merged_clean.csv and ../data/processed/test_merged_clean.csv


In [408]:
# We need to group by the calendar year so rolling windows and lags
# do not leak across the Jul-Oct gap into the next year.
train_clean_df["season_year"]= train_clean_df["date"].dt.year
test_clean_df["season_year"]= test_clean_df["date"].dt.year

In [409]:
def fill_missing_by_season(df, cols):
    df = df.sort_values(["season_year", "date"]).copy()
    df[cols] = df.groupby("season_year")[cols].ffill().bfill()
    return df

feature_cols = [c for c in train_clean_df.columns if c not in ["date", target_col, "season_year"]]
train_clean_df = fill_missing_by_season(train_clean_df, feature_cols)
test_clean_df  = fill_missing_by_season(test_clean_df, feature_cols)

In [410]:
print("=== TEST DATA MISSINGNESS ===")
# % missing per column
missing_pct = test_clean_df.isna().mean() * 100
missing_cols = missing_pct[missing_pct > 0].sort_values(ascending=False)

# Rows with ANY missing data
rows_with_missing = test_clean_df.isna().any(axis=1).sum()
pct_rows_missing = (rows_with_missing / len(test_clean_df)) * 100

if len(missing_cols) == 0:
    print("Columns with missing data: None! 100% complete.")
else:
    print(f"Columns with missing data:\n{missing_cols}")
print(f"Rows with at least one missing value: {rows_with_missing} ({pct_rows_missing:.2f}%)\n")

=== TEST DATA MISSINGNESS ===
Columns with missing data:
sst_cameroon_mean_temperature_deg_c              16.67
sst_cameroon_mean_temperature_uncertainty        16.67
sst_indian_ocean_mean_temperature_deg_c          16.67
sst_indian_ocean_mean_temperature_uncertainty    16.67
dtype: float64
Rows with at least one missing value: 123 (16.67%)



In [411]:
train_clean_df.head(10)

,date,5cm_soil_moist,mean_dew_point_temp,max_temp,sea_level_pressure,sst_cameroon_mean_temperature_deg_c,sst_cameroon_mean_temperature_uncertainty,sst_indian_ocean_mean_temperature_deg_c,sst_indian_ocean_mean_temperature_uncertainty,potential_water_deficit,2m_temp,u10,vapor_pressure_deficit,dryspell_warn_7d,year,month,season_year
0,2002-07-01,0.14,16.31,40.13,101.01,26.06,0.19,28.84,0.12,-4.61,32.73,0.76,5.45,0,2002,7,2002
1,2002-07-02,0.14,16.77,41.36,100.81,26.10,0.19,28.85,0.12,-2.92,33.37,0.68,5.52,0,2002,7,2002
2,2002-07-03,0.14,19.72,38.05,100.97,26.11,0.20,28.82,0.12,7.59,31.57,1.41,4.23,0,2002,7,2002
3,2002-07-04,0.14,18.50,39.51,100.94,26.14,0.23,28.79,0.12,-2.29,32.59,1.49,4.68,0,2002,7,2002
4,2002-07-05,0.14,17.68,40.47,100.86,26.29,0.21,28.78,0.12,-2.87,33.59,1.07,5.15,0,2002,7,2002
5,2002-07-06,0.14,16.01,39.39,100.80,26.19,0.21,28.74,0.12,-4.20,33.54,1.51,5.00,0,2002,7,2002
6,2002-07-07,0.13,17.04,39.07,100.77,26.13,0.20,28.63,0.13,-2.75,32.71,2.27,4.65,0,2002,7,2002
7,2002-07-08,0.13,16.27,38.14,100.86,26.03,0.21,28.55,0.12,6.13,32.13,0.97,4.58,0,2002,7,2002
8,2002-07-09,0.16,16.65,37.88,101.03,25.80,0.20,28.55,0.13,-3.01,31.17,0.73,4.73,0,2002,7,2002
9,2002-07-10,0.18,18.00,36.04,101.02,25.67,0.21,28.55,0.13,-2.68,30.29,0.97,4.07,0,2002,7,2002


In [412]:
test_clean_df.head()

,date,5cm_soil_moist,mean_dew_point_temp,max_temp,sea_level_pressure,sst_cameroon_mean_temperature_deg_c,sst_cameroon_mean_temperature_uncertainty,sst_indian_ocean_mean_temperature_deg_c,sst_indian_ocean_mean_temperature_uncertainty,potential_water_deficit,2m_temp,u10,vapor_pressure_deficit,year,month,season_year
0,2020-07-01,0.25,16.26,39.03,100.67,25.38,0.24,30.06,0.12,-4.10,32.32,3.13,4.96,2020,7,2020
1,2020-07-02,0.23,18.46,36.96,100.79,25.45,0.23,30.09,0.12,-3.35,31.19,3.37,4.15,2020,7,2020
2,2020-07-03,0.22,16.23,37.11,100.84,25.49,0.21,30.07,0.12,-4.79,30.94,1.36,4.56,2020,7,2020
3,2020-07-04,0.21,16.36,39.22,100.84,25.73,0.22,30.03,0.12,-1.23,31.98,1.60,5.08,2020,7,2020
4,2020-07-05,0.21,16.59,37.46,100.68,25.92,0.23,29.95,0.12,-3.73,31.39,1.56,4.23,2020,7,2020


In [413]:
# We keep only numeric columns for feature engineering.
# We exclude:
# - date: used only for ordering/splitting
# - target: label, never used as input
# - season_year: grouping helper, not a real predictor

exclude_cols =["date", target_col, "season_year"]

numeric_cols = train_clean_df.select_dtypes(include=[np.number]).columns.tolist()
base_features = [col for col in numeric_cols if col not in exclude_cols + ["year", "month"]]

print("Number of base numeric features:", len(base_features))
print(base_features)

Number of base numeric features: 12
['5cm_soil_moist', 'mean_dew_point_temp', 'max_temp', 'sea_level_pressure', 'sst_cameroon_mean_temperature_deg_c', 'sst_cameroon_mean_temperature_uncertainty', 'sst_indian_ocean_mean_temperature_deg_c', 'sst_indian_ocean_mean_temperature_uncertainty', 'potential_water_deficit', '2m_temp', 'u10', 'vapor_pressure_deficit']


In [414]:
rolling_windows = [7, 14, 30]

def add_rolling_features(df, feature_cols, windows):
    df = df.copy()
    for col in feature_cols:
        for window in windows:
            grouped = df.groupby("season_year")[col]
            df[f"{col}_rollmean_{window}"] = grouped.transform(lambda x: x.rolling(window, min_periods=window).mean())
            df[f"{col}_rollmin_{window}"]  = grouped.transform(lambda x: x.rolling(window, min_periods=window).min())
            df[f"{col}_rollmax_{window}"]  = grouped.transform(lambda x: x.rolling(window, min_periods=window).max())
    return df
        

In [415]:
train_feat = add_rolling_features(train_clean_df, base_features, rolling_windows)
test_feat = add_rolling_features(test_clean_df, base_features, rolling_windows)

print("Rolling features added.")
print("Train shape:", train_feat.shape)
print("Test shape:", test_feat.shape)

Rolling features added.
Train shape: (2214, 125)
Test shape: (738, 124)


In [416]:
train_feat.head()

,date,5cm_soil_moist,mean_dew_point_temp,max_temp,sea_level_pressure,sst_cameroon_mean_temperature_deg_c,sst_cameroon_mean_temperature_uncertainty,sst_indian_ocean_mean_temperature_deg_c,sst_indian_ocean_mean_temperature_uncertainty,potential_water_deficit,2m_temp,u10,vapor_pressure_deficit,dryspell_warn_7d,year,month,season_year,5cm_soil_moist_rollmean_7,5cm_soil_moist_rollmin_7,5cm_soil_moist_rollmax_7,5cm_soil_moist_rollmean_14,5cm_soil_moist_rollmin_14,5cm_soil_moist_rollmax_14,5cm_soil_moist_rollmean_30,5cm_soil_moist_rollmin_30,5cm_soil_moist_rollmax_30,mean_dew_point_temp_rollmean_7,mean_dew_point_temp_rollmin_7,mean_dew_point_temp_rollmax_7,mean_dew_point_temp_rollmean_14,mean_dew_point_temp_rollmin_14,mean_dew_point_temp_rollmax_14,mean_dew_point_temp_rollmean_30,mean_dew_point_temp_rollmin_30,mean_dew_point_temp_rollmax_30,max_temp_rollmean_7,max_temp_rollmin_7,max_temp_rollmax_7,max_temp_rollmean_14,max_temp_rollmin_14,max_temp_rollmax_14,max_temp_rollmean_30,max_temp_rollmin_30,max_temp_rollmax_30,sea_level_pressure_rollmean_7,sea_level_pressure_rollmin_7,sea_level_pressure_rollmax_7,sea_level_pressure_rollmean_14,sea_level_pressure_rollmin_14,sea_level_pressure_rollmax_14,sea_level_pressure_rollmean_30,sea_level_pressure_rollmin_30,sea_level_pressure_rollmax_30,sst_cameroon_mean_temperature_deg_c_rollmean_7,sst_cameroon_mean_temperature_deg_c_rollmin_7,sst_cameroon_mean_temperature_deg_c_rollmax_7,sst_cameroon_mean_temperature_deg_c_rollmean_14,sst_cameroon_mean_temperature_deg_c_rollmin_14,sst_cameroon_mean_temperature_deg_c_rollmax_14,sst_cameroon_mean_temperature_deg_c_rollmean_30,sst_cameroon_mean_temperature_deg_c_rollmin_30,sst_cameroon_mean_temperature_deg_c_rollmax_30,sst_cameroon_mean_temperature_uncertainty_rollmean_7,sst_cameroon_mean_temperature_uncertainty_rollmin_7,sst_cameroon_mean_temperature_uncertainty_rollmax_7,sst_cameroon_mean_temperature_uncertainty_rollmean_14,sst_cameroon_mean_temperature_uncertainty_rollmin_14,sst_cameroon_mean_temperature_uncertainty_rollmax_14,sst_cameroon_mean_temperature_uncertainty_rollmean_30,sst_cameroon_mean_temperature_uncertainty_rollmin_30,sst_cameroon_mean_temperature_uncertainty_rollmax_30,sst_indian_ocean_mean_temperature_deg_c_rollmean_7,sst_indian_ocean_mean_temperature_deg_c_rollmin_7,sst_indian_ocean_mean_temperature_deg_c_rollmax_7,sst_indian_ocean_mean_temperature_deg_c_rollmean_14,sst_indian_ocean_mean_temperature_deg_c_rollmin_14,sst_indian_ocean_mean_temperature_deg_c_rollmax_14,sst_indian_ocean_mean_temperature_deg_c_rollmean_30,sst_indian_ocean_mean_temperature_deg_c_rollmin_30,sst_indian_ocean_mean_temperature_deg_c_rollmax_30,sst_indian_ocean_mean_temperature_uncertainty_rollmean_7,sst_indian_ocean_mean_temperature_uncertainty_rollmin_7,sst_indian_ocean_mean_temperature_uncertainty_rollmax_7,sst_indian_ocean_mean_temperature_uncertainty_rollmean_14,sst_indian_ocean_mean_temperature_uncertainty_rollmin_14,sst_indian_ocean_mean_temperature_uncertainty_rollmax_14,sst_indian_ocean_mean_temperature_uncertainty_rollmean_30,sst_indian_ocean_mean_temperature_uncertainty_rollmin_30,sst_indian_ocean_mean_temperature_uncertainty_rollmax_30,potential_water_deficit_rollmean_7,potential_water_deficit_rollmin_7,potential_water_deficit_rollmax_7,potential_water_deficit_rollmean_14,potential_water_deficit_rollmin_14,potential_water_deficit_rollmax_14,potential_water_deficit_rollmean_30,potential_water_deficit_rollmin_30,potential_water_deficit_rollmax_30,2m_temp_rollmean_7,2m_temp_rollmin_7,2m_temp_rollmax_7,2m_temp_rollmean_14,2m_temp_rollmin_14,2m_temp_rollmax_14,2m_temp_rollmean_30,2m_temp_rollmin_30,2m_temp_rollmax_30,u10_rollmean_7,u10_rollmin_7,u10_rollmax_7,u10_rollmean_14,u10_rollmin_14,u10_rollmax_14,u10_rollmean_30,u10_rollmin_30,u10_rollmax_30,vapor_pressure_deficit_rollmean_7,vapor_pressure_deficit_rollmin_7,vapor_pressure_deficit_rollmax_7,vapor_pressure_deficit_rollmean_14,vapor_pressure_deficit_rollmin_14,vapor_pressure_deficit_rollmax_14,vapor_pre

In [417]:
# Define key signals for lag features
lag_features = [
    "vapor_pressure_deficit",
    "5cm_soil_moist",
    "max_temp",
    "mean_dew_point_temp",
    "u10",
    "sea_level_pressure",
    "potential_water_deficit",
    "sst_cameroon_mean_temperature_deg_c",
    "sst_indian_ocean_mean_temperature_deg_c",
]
# Keep only columns that actually exist in the dataframe
lag_features = [col for col in lag_features if col in train_feat.columns]

print("Lag feature columns found:")
print(lag_features)

Lag feature columns found:
['vapor_pressure_deficit', '5cm_soil_moist', 'max_temp', 'mean_dew_point_temp', 'u10', 'sea_level_pressure', 'potential_water_deficit', 'sst_cameroon_mean_temperature_deg_c', 'sst_indian_ocean_mean_temperature_deg_c']


In [418]:
# Create lag features and diff_1 / diff_7
lags = [1, 7, 14]

def add_lag_features(df, feature_cols, lags):
    df = df.copy()
    
    for col in feature_cols:
        grouped = df.groupby("season_year")[col]
        
        for lag in lags:
            df[f"{col}_lag_{lag}"] = grouped.transform(lambda x: x.shift(lag))
        
        # Short-term change: today minus yesterday
        if 1 in lags:
            df[f"{col}_diff_1"] = df[col] - df[f"{col}_lag_1"]
        
        # Trend proxy: today minus 7 days ago
        if 7 in lags:
            df[f"{col}_diff_7"] = df[col] - df[f"{col}_lag_7"]
    
    return df


train_feat = add_lag_features(train_feat, lag_features, lags)
test_feat = add_lag_features(test_feat, lag_features, lags)

print("Lag and diff features added.")
print("Train shape:", train_feat.shape)
print("Test shape:", test_feat.shape)

Lag and diff features added.
Train shape: (2214, 170)
Test shape: (738, 169)


In [419]:
 # Drop early-season rows with incomplete rolling/lag features

# Since rolling window max = 30, the first 29 rows of each season
# will not have full rolling statistics.
# We drop rows with any NA in engineered numeric columns.

def drop_engineering_na(df):
    df = df.copy()
    return df.dropna().reset_index(drop=True)

train_feat = drop_engineering_na(train_feat)
test_feat = drop_engineering_na(test_feat)

print("After dropping NA rows:")
print("Train shape:", train_feat.shape)
print("Test shape:", test_feat.shape)

After dropping NA rows:
Train shape: (1692, 170)
Test shape: (470, 169)


In [420]:
train_feat.head()

,date,5cm_soil_moist,mean_dew_point_temp,max_temp,sea_level_pressure,sst_cameroon_mean_temperature_deg_c,sst_cameroon_mean_temperature_uncertainty,sst_indian_ocean_mean_temperature_deg_c,sst_indian_ocean_mean_temperature_uncertainty,potential_water_deficit,2m_temp,u10,vapor_pressure_deficit,dryspell_warn_7d,year,month,season_year,5cm_soil_moist_rollmean_7,5cm_soil_moist_rollmin_7,5cm_soil_moist_rollmax_7,5cm_soil_moist_rollmean_14,5cm_soil_moist_rollmin_14,5cm_soil_moist_rollmax_14,5cm_soil_moist_rollmean_30,5cm_soil_moist_rollmin_30,5cm_soil_moist_rollmax_30,mean_dew_point_temp_rollmean_7,mean_dew_point_temp_rollmin_7,mean_dew_point_temp_rollmax_7,mean_dew_point_temp_rollmean_14,mean_dew_point_temp_rollmin_14,mean_dew_point_temp_rollmax_14,mean_dew_point_temp_rollmean_30,mean_dew_point_temp_rollmin_30,mean_dew_point_temp_rollmax_30,max_temp_rollmean_7,max_temp_rollmin_7,max_temp_rollmax_7,max_temp_rollmean_14,max_temp_rollmin_14,max_temp_rollmax_14,max_temp_rollmean_30,max_temp_rollmin_30,max_temp_rollmax_30,sea_level_pressure_rollmean_7,sea_level_pressure_rollmin_7,sea_level_pressure_rollmax_7,sea_level_pressure_rollmean_14,sea_level_pressure_rollmin_14,sea_level_pressure_rollmax_14,sea_level_pressure_rollmean_30,sea_level_pressure_rollmin_30,sea_level_pressure_rollmax_30,sst_cameroon_mean_temperature_deg_c_rollmean_7,sst_cameroon_mean_temperature_deg_c_rollmin_7,sst_cameroon_mean_temperature_deg_c_rollmax_7,sst_cameroon_mean_temperature_deg_c_rollmean_14,sst_cameroon_mean_temperature_deg_c_rollmin_14,sst_cameroon_mean_temperature_deg_c_rollmax_14,sst_cameroon_mean_temperature_deg_c_rollmean_30,sst_cameroon_mean_temperature_deg_c_rollmin_30,sst_cameroon_mean_temperature_deg_c_rollmax_30,sst_cameroon_mean_temperature_uncertainty_rollmean_7,sst_cameroon_mean_temperature_uncertainty_rollmin_7,sst_cameroon_mean_temperature_uncertainty_rollmax_7,sst_cameroon_mean_temperature_uncertainty_rollmean_14,sst_cameroon_mean_temperature_uncertainty_rollmin_14,sst_cameroon_mean_temperature_uncertainty_rollmax_14,sst_cameroon_mean_temperature_uncertainty_rollmean_30,sst_cameroon_mean_temperature_uncertainty_rollmin_30,sst_cameroon_mean_temperature_uncertainty_rollmax_30,sst_indian_ocean_mean_temperature_deg_c_rollmean_7,sst_indian_ocean_mean_temperature_deg_c_rollmin_7,sst_indian_ocean_mean_temperature_deg_c_rollmax_7,sst_indian_ocean_mean_temperature_deg_c_rollmean_14,sst_indian_ocean_mean_temperature_deg_c_rollmin_14,sst_indian_ocean_mean_temperature_deg_c_rollmax_14,sst_indian_ocean_mean_temperature_deg_c_rollmean_30,sst_indian_ocean_mean_temperature_deg_c_rollmin_30,sst_indian_ocean_mean_temperature_deg_c_rollmax_30,sst_indian_ocean_mean_temperature_uncertainty_rollmean_7,sst_indian_ocean_mean_temperature_uncertainty_rollmin_7,sst_indian_ocean_mean_temperature_uncertainty_rollmax_7,sst_indian_ocean_mean_temperature_uncertainty_rollmean_14,sst_indian_ocean_mean_temperature_uncertainty_rollmin_14,sst_indian_ocean_mean_temperature_uncertainty_rollmax_14,sst_indian_ocean_mean_temperature_uncertainty_rollmean_30,sst_indian_ocean_mean_temperature_uncertainty_rollmin_30,sst_indian_ocean_mean_temperature_uncertainty_rollmax_30,potential_water_deficit_rollmean_7,potential_water_deficit_rollmin_7,potential_water_deficit_rollmax_7,potential_water_deficit_rollmean_14,potential_water_deficit_rollmin_14,potential_water_deficit_rollmax_14,potential_water_deficit_rollmean_30,potential_water_deficit_rollmin_30,potential_water_deficit_rollmax_30,2m_temp_rollmean_7,2m_temp_rollmin_7,2m_temp_rollmax_7,2m_temp_rollmean_14,2m_temp_rollmin_14,2m_temp_rollmax_14,2m_temp_rollmean_30,2m_temp_rollmin_30,2m_temp_rollmax_30,u10_rollmean_7,u10_rollmin_7,u10_rollmax_7,u10_rollmean_14,u10_rollmin_14,u10_rollmax_14,u10_rollmean_30,u10_rollmin_30,u10_rollmax_30,vapor_pressure_deficit_rollmean_7,vapor_pressure_deficit_rollmin_7,vapor_pressure_deficit_rollmax_7,vapor_pressure_deficit_rollmean_14,vapor_pressure_deficit_rollmin_14,vapor_pressure_deficit_rollmax_14,vapor_pre

In [421]:
train_feat.to_csv("../data/processed/train_features.csv", index=False)
test_feat.to_csv("../data/processed/test_features.csv", index=False)

print("Feature-engineered datasets saved successfully.")

Feature-engineered datasets saved successfully.
